# Module 1 · Lesson 02: Your First OpenAI API Call

Welcome! In this notebook you will learn how to **communicate with OpenAI's GPT models** through their API.

## What you will learn
1. How to initialize the OpenAI client
2. The anatomy of a **chat completion** request
3. Using **system prompts** to control behaviour
4. How **temperature** affects creativity
5. Building **multi-turn conversations**

---

### Prerequisites
Make sure you have run `01_setup_verification.py` and that your `OPENAI_API_KEY` is set in `.env`.

OpenAI API Key: https://platform.openai.com/api-keys

OpenAI API Pricing: https://openai.com/api/pricing/

In [4]:
import os

from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI

client = OpenAI()

if client:
    display(Markdown("Client Ready! ✅"))

Client Ready! ✅

---
## 1. Basic Completion — Your Very First Call

The `chat.completions.create()` method is the **core building block** of every LLM application.

Three required parameters:
| Parameter | Purpose |
|-----------|--------|
| `model`   | Which GPT model to use |
| `messages` | The conversation so far (list of dicts) |
| `max_tokens` | Maximum length of the response |

Each message has a `role` (`"system"`, `"user"`, or `"assistant"`) and `content`.

In [10]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user",
         "content": "What is Python? Answer in one sentence."}
    ],
    max_tokens=200
)

# extract answer
#print(response)
answer = response.choices[0].message.content
#print(answer)
display(Markdown(f"**Answer:** {answer}"))

#Token usage
u = response.usage
print(f"Tokens - Promt: {u.prompt_tokens}, completion: {u.completion_tokens}, total: {u.total_tokens}")

**Answer:** Python is a high-level, interpreted programming language known for its readability, versatility, and extensive libraries, making it popular for a wide range of applications from web development to data analysis.

Tokens - Promt: 16, completion: 36, total: 52


> **Key Insight:** The API is *stateless* — it does not remember previous calls.
> Every request must contain the full conversation context.

---
## 2. System Prompts — Controlling Behaviour

A **system prompt** sets the AI's persona, tone, and constraints *before* the user's message.
Think of it as the "instruction manual" the model follows.

In [11]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful programmins tutor. "
                "Explain concepts simply using analogies."
                "Keep responses concise (max 3 sentences)"
            )
        },
        {
            "role": "user",
            "content": "What is recursion?"
        }
    ],
    max_tokens=500
)

display(Markdown(f"### Tutor Response\n\n {response.choices[0].message.content}"))

### Tutor Response

 Recursion is like a Russian nesting doll: each doll can open up to reveal a smaller doll inside. In programming, a recursive function calls itself with a simpler version of the original problem until it reaches a basic case that can be solved directly. Just like you keep opening dolls until you find the tiniest one, the function keeps calling itself until it reaches the simplest case.

> **Best Practice:** Always include a system prompt in production.
> It improves consistency, safety, and output quality.

---
## 3. Temperature — Creativity vs Determinism

| Temperature | Behaviour | Use Case |
|-------------|-----------|----------|
| `0.0` | Deterministic, repeatable | Classification, extraction, math |
| `0.3–0.7` | Balanced | General Q&A, summarisation |
| `1.0` | Creative, varied | Brainstorming, storytelling |

Let's compare:

In [17]:
prompt = "Γράψε μου μια ιστορία, το πολύ 2 προτάσεις, για ένα ρομπότ που ήθελε να μάθει Python"

for temp in [0.0, 1.0]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temp,
        max_tokens=300
    )
    label = "Deterministic" if temp == 0 else "Creative"
display(Markdown(f"**Temperature {temp} ({label}):** {response.choices[0].message.content}"))

**Temperature 1.0 (Creative):** Ένα μικρό ρομπότ ονόματι Ρόμπι αποφάσισε να μάθει Python για να μπορούσε να δημιουργεί μοναδικά παιχνίδια και ιστορίες. Με κάθε νέα γραμμή κώδικα, ανακάλυπτε όχι μόνο τις δυνατότητές του, αλλά και τη χαρά της δημιουργίας.

---
## The Problem: LLMs Have NO Memory!

Before we learn the multi-turn pattern, let's **prove** that the API has **no memory**.

Each API call is completely independent. The model doesn't know what you asked before.
Watch what happens when we make two separate calls:

In [18]:
prompt1 = "My namme is Georgios and I am a software engineer."

response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt1}
    ],
    max_tokens=100
)

print("- - - Call 1 - - -")
print(f"User: {prompt1}")
print(f"Assistant: {response1.choices[0].message.content}")

- - - Call 1 - - -
User: My namme is Georgios and I am a software engineer.
Assistant: Nice to meet you, Georgios! As a software engineer, what specific areas do you focus on? Are there any projects you're currently working on or technologies you're particularly interested in?


In [19]:
prompt2 = "What is my name and wahat do i do?"

response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt2}
    ],
    max_tokens=100
)
print("- - - Call 2 - - -")
print(f"User: {prompt2}")
print(f"Assistant: {response2.choices[0].message.content}")

- - - Call 2 - - -
User: What is my name and wahat do i do?
Assistant: I'm sorry, but I don't know your name or what you do. If you share that information with me, I'd be happy to engage in a conversation with you!


In [20]:
prompt3 = "Τί καιρό κάνει σήμερα στην Αθήνα;"

response3 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt3}
    ],
    max_tokens=100
)
print("- - - Call 3 - - -")
print(f"User: {prompt3}")
print(f"Assistant: {response3.choices[0].message.content}") 

- - - Call 3 - - -
User: Τί καιρό κάνει σήμερα στην Αθήνα;
Assistant: Λυπάμαι, αλλά δεν μπορώ να παρέχω πληροφορίες για τον τρέχοντα καιρό. Μπορείτε να ελέγξετε μια αξιόπιστη ιστοσελίδα ή εφαρμογή καιρού για να μάθετε τις τρέχουσες συνθήκες στην Αθήνα.


---
## 4. Multi-Turn Conversations

Since the API is **stateless**, we must send the full conversation history every time.
The pattern is:

```
messages = [
    {"role": "system", "content": "..."},   # Instructions
    {"role": "user",   "content": "..."},    # User turn 1
    {"role": "assistant", "content": "..."},  # Model reply 1
    {"role": "user",   "content": "..."},    # User turn 2
    ...                                        # And so on
]
```

In [23]:
prompt1 = "My name is Georgios"
messages = [
    {"role": "user", "content": "You are a helpful assistant. Be concise."},
    {"role": "user", "content": prompt1}
]

# Turn 1
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    max_tokens=100
)

reply1 = response1.choices[0].message.content
print(f"User: {prompt1}")
print(f"Assistant: {reply1} \n")

# Add assistant's reply to the conversation history
messages.append({"role": "assistant", "content": reply1})

# Turn 2
prompt2 = "What is my name?"
messages.append({"role": "user", "content": prompt2})
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    max_tokens=50
)
print(f"User: {prompt2}")
print(f"Assistant: {response2.choices[0].message.content}")

User: My name is Georgios
Assistant: Nice to meet you, Georgios! How can I assist you today? 

User: What is my name?
Assistant: Your name is Georgios.


---
## Streaming Responses

By default, the API waits until the **entire** response is generated before returning it.
With **streaming**, tokens arrive one by one -- just like ChatGPT's typing effect!

| Mode | Behaviour | Use Case |
|------|-----------|----------|
| Normal | Wait for full response | Background jobs, data extraction |
| Streaming | Tokens arrive live | Chat UIs, real-time applications |

To enable streaming, simply add `stream=True`:

In [29]:
import time

prompt = "Explain what streaming means in 3 sentences."
stream = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": prompt}
    ],
    max_tokens= 250,
    stream=True # enbale streaming.
)

full_text = ""
start = time.time()

for chunk in stream:
    token = chunk.choices[0].delta.content
    if token:
        print(token, end="", flush=True)
        full_text += token

elapsed = time.time() - start

print(f"Total: {len(full_text)} chars in {elapsed:.1f} s")
print(f"First token appeared almost instantly: {elapsed:.1f} s for full response")


Streaming refers to the process of delivering audio, video, or other media content over the internet in real-time, allowing users to access it without downloading the entire file. This technology enables on-demand access, meaning users can watch or listen to content as it is being transmitted. Popular examples include services like Netflix for video and Spotify for music.Total: 374 chars in 2.3 s
First token appeared almost instantly: 2.3 s for full response


---
## 5. Exercise — Try It Yourself!

Modify the cell below to:
1. Change the **system prompt** to a different persona (e.g., "You are a Shakespearean poet")
2. Ask the model a question
3. Experiment with different `temperature` values

---
## Key Takeaways

| Concept | Summary |
|---------|--------|
| **Client** | `OpenAI()` reads the API key from environment automatically |
| **Messages** | List of `{role, content}` dicts -- system, user, assistant |
| **System Prompt** | Sets behaviour/persona -- always include in production |
| **Temperature** | 0 = deterministic, 1 = creative |
| **No Memory** | Separate calls do NOT share context -- you must send the full history |
| **Multi-Turn** | Append assistant replies to `messages` list to maintain conversation |
| **Streaming** | `stream=True` makes tokens arrive one by one for responsive UIs |
| **max_tokens** | Controls maximum response length (and cost!) |

---
**Next:** `03_first_anthropic_call.ipynb` -- Learn Claude's API and compare with OpenAI